In [1]:
import spacy

nlp = spacy.load('en_core_web_lg')

In [2]:
doc = nlp("dog cat banana asdfghds")

for token in doc:
    print(token,'|',token.has_vector,'|',token.is_oov)

dog | True | False
cat | True | False
banana | True | False
asdfghds | False | True


In [3]:
doc[0].vector

array([-4.0176e-01,  3.7057e-01,  2.1281e-02, -3.4125e-01,  4.9538e-02,
        2.9440e-01, -1.7376e-01, -2.7982e-01,  6.7622e-02,  2.1693e+00,
       -6.2691e-01,  2.9106e-01, -6.7270e-01,  2.3319e-01, -3.4264e-01,
        1.8311e-01,  5.0226e-01,  1.0689e+00,  1.4698e-01, -4.5230e-01,
       -4.1827e-01, -1.5967e-01,  2.6748e-01, -4.8867e-01,  3.6462e-01,
       -4.3403e-02, -2.4474e-01, -4.1752e-01,  8.9088e-02, -2.5552e-01,
       -5.5695e-01,  1.2243e-01, -8.3526e-02,  5.5095e-01,  3.6410e-01,
        1.5361e-01,  5.5738e-01, -9.0702e-01, -4.9098e-02,  3.8580e-01,
        3.8000e-01,  1.4425e-01, -2.7221e-01, -3.7016e-01, -1.2904e-01,
       -1.5085e-01, -3.8076e-01,  4.9583e-02,  1.2755e-01, -8.2788e-02,
        1.4339e-01,  3.2537e-01,  2.7226e-01,  4.3632e-01, -3.1769e-01,
        7.9405e-01,  2.6529e-01,  1.0135e-01, -3.3279e-01,  4.3117e-01,
        1.6687e-01,  1.0729e-01,  8.9418e-02,  2.8635e-01,  4.0117e-01,
       -3.9222e-01,  4.5217e-01,  1.3521e-01, -2.8878e-01, -2.28

In [4]:
doc[0].vector.shape

(300,)

In [5]:
base_token = nlp('bread')
doc = nlp("bread sandwich burger car tiger human wheat")

for token in doc:
    print(f"{token.text} <-> {base_token.text}: {token.similarity(base_token)}")

bread <-> bread: 0.9999999766167111
sandwich <-> bread: 0.6874560014053445
burger <-> bread: 0.5440373883702087
car <-> bread: 0.1644114584391833
tiger <-> bread: 0.1449235625942581
human <-> bread: 0.21103660928832707
wheat <-> bread: 0.6572456428272563


In [6]:
def print_similarity(base_word, words_to_compare):
    base_token = nlp(base_word)
    doc = nlp(words_to_compare)

    for token in doc:
        print(f"{token.text} <-> {base_token.text}: {token.similarity(base_token)}")

In [7]:
print_similarity('iphone',"apple samsung iphone dog kitten")

apple <-> iphone: 0.6339781147910419
samsung <-> iphone: 0.6678678014329177
iphone <-> iphone: 1.0000000285783557
dog <-> iphone: 0.17431037640553934
kitten <-> iphone: 0.14685812907484028


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
king = nlp.vocab['king'].vector
man = nlp.vocab['man'].vector
woman = nlp.vocab['woman'].vector
queen = nlp.vocab['queen'].vector

result = king - man + woman
print(cosine_similarity([queen],[result]).item())

0.7880844473838806


In [9]:
import pandas as pd

df = pd.read_csv('Data Files/Fake_Real_Data.csv')
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [10]:
df.shape

(9900, 2)

In [11]:
df['label'].value_counts()

label
Fake    5000
Real    4900
Name: count, dtype: int64

In [13]:
df['target'] = df['label'].apply(lambda x: 1 if x == 'Fake' else 0)

In [14]:
df.head()

,Text,label,target
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,1
1,U.S. conservative leader optimistic of common ...,Real,0
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,0
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,1
4,Democrats say Trump agrees to work on immigrat...,Real,0


In [16]:
df['vector'] = df['Text'].apply(lambda x: nlp(x).vector)
df.head()

,Text,label,target,vector
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,1,"[-0.103623025, 0.17802684, -0.11873861, -0.034..."
1,U.S. conservative leader optimistic of common ...,Real,0,"[-0.0063406364, 0.16712041, -0.06661373, 0.017..."
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,0,"[-0.122753024, 0.17192385, -0.024732638, -0.06..."
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,1,"[-0.027337318, 0.12501417, -0.0073965387, -0.0..."
4,Democrats say Trump agrees to work on immigrat...,Real,0,"[-0.032708026, 0.093958504, -0.03287002, -0.00..."


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['vector'].values,df['target'], test_size=0.2, random_state=2025, stratify=df['target'])

In [18]:
import numpy as np

X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)
X_train_2d.shape

(7920, 300)

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

scalar = MinMaxScaler()
scaled_train_embed = scalar.fit_transform(X_train_2d)
scaled_test_embed = scalar.transform(X_test_2d)

clf = MultinomialNB()

clf.fit(scaled_train_embed,y_train)

MultinomialNB()

In [21]:
from sklearn.metrics import classification_report

y_pred = clf.predict(scaled_test_embed)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94       980
           1       0.93      0.96      0.94      1000

    accuracy                           0.94      1980
   macro avg       0.94      0.94      0.94      1980
weighted avg       0.94      0.94      0.94      1980



In [23]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')

knn.fit(X_train_2d,y_train)
y_pred = knn.predict(X_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       980
           1       0.98      0.97      0.97      1000

    accuracy                           0.97      1980
   macro avg       0.97      0.97      0.97      1980
weighted avg       0.97      0.97      0.97      1980



### Exercise

In [24]:
import pandas as pd

df = pd.read_json('Data Files/news_dataset.json',lines=False)

print(df.info(),'\n\n',df.describe())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 12695 entries, 0 to 12694
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      12695 non-null  object
 1   category  12695 non-null  object
dtypes: object(2)
memory usage: 297.5+ KB
None 

                                                      text  category
count                                               12695     12695
unique                                              12689         4
top     10 Most Hated Companies In America To be truly...  BUSINESS
freq                                                    2      4254
                                                text  category
0  Watching Schrödinger's Cat Die University of C...   SCIENCE
1     WATCH: Freaky Vortex Opens Up In Flooded Lake    SCIENCE
2  Entrepreneurs Today Don't Need a Big Budget to...  BUSINESS
3  These Roads Could Recharge Your Electric Car A...  BUSINESS
4  Civilian 'Guard' Fires Gun While 'Pro

In [25]:
df['category'].value_counts()

category
BUSINESS    4254
SPORTS      4167
CRIME       2893
SCIENCE     1381
Name: count, dtype: int64